# YOLO Hyperparameter Tuning

This notebook demonstrates how to perform hyperparameter tuning for YOLO training using Optuna in a sequential manner. This approach prevents Out-Of-Memory (OOM) errors and ensures the best combination of parameters are identified for our rock cutting dataset.

In [ ]:
import os
import gc
import torch
import optuna
from ultralytics import YOLO
from pathlib import Path
import random
import string

# Prevent memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

VERSION = "Batch3and4_YOLO"
DATASET_ROOT = Path("../../datasets") / VERSION
DATASET_YAML = DATASET_ROOT / "data.yaml"

YOLO_MODEL = "yolo26m-seg.pt"
IMG_SIZE = 1024
BATCH_SIZE = 16

def generate_runner_name(prefix="Tuning"):
    suffix = ''.join(random.choices(string.ascii_letters + string.digits, k=11))
    return f"{prefix}_{suffix}"

TUNE_NAME = generate_runner_name()

In [ ]:
def run_optuna_tuning():
    print("\n" + "=" * 80)
    print("STARTING HYPERPARAMETER TUNING (OPTUNA)")
    print("=" * 80 + "\n")

    TUNE_EPOCHS = 30
    TUNE_ITERATIONS = 10
    
    # Safe batch sizing for memory constrained tuning
    tune_batch_size = max(2, BATCH_SIZE // 4)
    print(f"Using severely reduced batch_size: {tune_batch_size} (prevents OOM)")

    def objective(trial):
        print(f"\nStarting Trial {trial.number}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        # Search Space
        lr0 = trial.suggest_float("lr0", 1e-5, 1e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 0.0, 0.001)
        hsv_s = trial.suggest_float("hsv_s", 0.3, 0.9)
        hsv_v = trial.suggest_float("hsv_v", 0.3, 0.9)
        cls_weight = trial.suggest_float("cls", 0.5, 2.0)
        box_weight = trial.suggest_float("box", 0.02, 0.1)
        mask_weight = trial.suggest_float("mask", 0.5, 2.0)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)
        overlap_mask = trial.suggest_categorical("overlap_mask", [True, False])
        
        model = YOLO(YOLO_MODEL)
        
        try:
            results = model.train(
                data=DATASET_YAML,
                epochs=TUNE_EPOCHS,
                imgsz=IMG_SIZE,
                batch=tune_batch_size,
                optimizer="AdamW",
                lr0=lr0,
                weight_decay=weight_decay,
                hsv_s=hsv_s,
                hsv_v=hsv_v,
                cls=cls_weight,
                box=box_weight,
                mask=mask_weight,
                dropout=dropout,
                overlap_mask=overlap_mask,
                project=str(DATASET_ROOT / "models" / TUNE_NAME),
                name=f"trial_{trial.number}",
                verbose=False,
                plots=False,
                save=False,
                workers=2,
                device=0 if torch.cuda.is_available() else 'cpu'
            )
            
            if hasattr(results, 'seg') and results.seg is not None:
                metric = results.seg.map
            else:
                metric = results.box.map
                
            return metric
            
        except Exception as e:
            print(f"Error during Trial {trial.number}: {str(e)}")
            raise optuna.exceptions.TrialPruned()
            
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    study = optuna.create_study(direction="maximize", study_name=TUNE_NAME)
    study.optimize(objective, n_trials=TUNE_ITERATIONS, gc_after_trial=True)
    
    print("\n" + "=" * 80)
    print("OPTUNA TUNING COMPLETED!")
    print("=" * 80)
    print("Best Hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  - {key}: {value}")

    return study.best_params

best_optuna_results = run_optuna_tuning()